# MouserCV - Notebook 4: Import / Export

This notebook provides bulk data management for MouserCV:

1. **Bulk upload** videos to GCS with auto-generated metadata
2. **List** all video metadata
3. **Bulk download** processed results
4. **Export** combined results to CSV / Excel
5. **Dataset summary** with statistics

**Runtime:** CPU is sufficient (no GPU needed).

## 1. Setup

In [ ]:
!pip install -q google-cloud-storage pydantic pandas openpyxl opencv-python-headless matplotlib

## 2. Authenticate and Configure

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully.')

In [ ]:
# ============================================================
# EDITABLE PARAMETERS
# ============================================================
BUCKET = "mousercv-data"
PROJECT = "demo"

# For bulk upload: upload all .mp4 files from this Colab directory.
# Upload files to Colab first, or mount Google Drive.
LOCAL_VIDEO_DIR = "/content/videos"  # Adjust as needed

## 3. Bulk Upload Videos

Iterate `.mp4` files in `LOCAL_VIDEO_DIR`, extract metadata with OpenCV,
generate a `video_id`, and upload both the video and its metadata to GCS.

In [ ]:
import os
import json
import cv2
from pathlib import Path
from datetime import datetime, timezone
from google.cloud import storage

gcs_client = storage.Client()
gcs_bucket = gcs_client.bucket(BUCKET)

video_dir = Path(LOCAL_VIDEO_DIR)
if not video_dir.exists():
    print(f"Directory {LOCAL_VIDEO_DIR} does not exist.")
    print("Upload .mp4 files to Colab or mount Google Drive first.")
else:
    mp4_files = sorted(video_dir.glob("*.mp4"))
    print(f"Found {len(mp4_files)} .mp4 files in {LOCAL_VIDEO_DIR}\n")

    for i, mp4_path in enumerate(mp4_files):
        video_id = mp4_path.stem  # filename without extension

        # Extract metadata with OpenCV
        cap = cv2.VideoCapture(str(mp4_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration_sec = frame_count / fps if fps > 0 else 0
        cap.release()

        # Upload video
        gcs_video_path = f"videos/{PROJECT}/{video_id}.mp4"
        blob = gcs_bucket.blob(gcs_video_path)
        blob.upload_from_filename(str(mp4_path))

        # Create metadata
        metadata = {
            "video_id": video_id,
            "filename": mp4_path.name,
            "project_name": PROJECT,
            "dataset_name": None,
            "subject_count": 2,
            "fps": fps,
            "duration_sec": round(duration_sec, 2),
            "width": width,
            "height": height,
            "camera_angle": "front_angled",
            "calibration_px_per_cm": None,
            "notes": None,
            "tags": [],
            "gcs_video_uri": f"gs://{BUCKET}/{gcs_video_path}",
            "gcs_results_uri": None,
            "processing_status": "uploaded",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "processed_at": None,
        }

        # Upload metadata
        gcs_meta_path = f"metadata/{video_id}.json"
        meta_blob = gcs_bucket.blob(gcs_meta_path)
        meta_blob.upload_from_string(
            json.dumps(metadata, indent=2),
            content_type="application/json",
        )

        size_mb = mp4_path.stat().st_size / (1024 * 1024)
        print(
            f"[{i + 1}/{len(mp4_files)}] {video_id}: "
            f"{width}x{height} @ {fps:.1f}fps, {duration_sec:.1f}s, "
            f"{size_mb:.1f} MB -> gs://{BUCKET}/{gcs_video_path}"
        )

    print(f"\nUpload complete: {len(mp4_files)} videos.")

## 4. List All Video Metadata

In [ ]:
import pandas as pd

# List all metadata/*.json blobs
meta_blobs = list(gcs_bucket.list_blobs(prefix="metadata/"))
meta_blobs = [b for b in meta_blobs if b.name.endswith(".json")]

print(f"Found {len(meta_blobs)} metadata files\n")

all_metadata = []
for blob in meta_blobs:
    content = blob.download_as_text()
    meta = json.loads(content)
    all_metadata.append(meta)

meta_df = pd.DataFrame(all_metadata)

# Display key columns
display_cols = [
    "video_id", "project_name", "subject_count", "fps",
    "duration_sec", "width", "height", "processing_status", "created_at",
]
available_cols = [c for c in display_cols if c in meta_df.columns]
meta_df[available_cols]

## 5. Bulk Download Processed Results

Download `behaviors.json` for all videos with `processing_status = "ready"`
and combine into a single DataFrame.

In [ ]:
processed = [m for m in all_metadata if m.get("processing_status") == "ready"]
print(f"Found {len(processed)} processed videos\n")

all_behavior_rows = []

for meta in processed:
    vid = meta["video_id"]
    gcs_path = f"results/{vid}/behaviors.json"
    blob = gcs_bucket.blob(gcs_path)

    if not blob.exists():
        print(f"  SKIP {vid}: no behaviors.json found")
        continue

    content = blob.download_as_text()
    segments = json.loads(content)

    for seg in segments:
        row = {
            "video_id": vid,
            "track_label": seg.get("track_label", "unknown"),
            "behavior": seg["behavior"],
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "duration_sec": round(seg["end_sec"] - seg["start_sec"], 3),
            "source": seg.get("source", "unknown"),
            "confidence": seg.get("confidence", 1.0),
        }
        all_behavior_rows.append(row)

    print(f"  {vid}: {len(segments)} segments")

if all_behavior_rows:
    results_df = pd.DataFrame(all_behavior_rows)
    print(f"\nCombined: {len(results_df)} behavior segments across "
          f"{results_df.video_id.nunique()} videos")
    results_df.head(10)
else:
    results_df = pd.DataFrame()
    print("No processed results found.")

## 6. Export to CSV / Excel

In [ ]:
if not results_df.empty:
    # CSV
    csv_path = "/tmp/mousercv_behaviors.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved CSV: {csv_path} ({os.path.getsize(csv_path) / 1024:.1f} KB)")

    # Excel
    xlsx_path = "/tmp/mousercv_behaviors.xlsx"
    results_df.to_excel(xlsx_path, index=False, sheet_name="Behaviors")
    print(f"Saved Excel: {xlsx_path} ({os.path.getsize(xlsx_path) / 1024:.1f} KB)")

    # Download links in Colab
    try:
        from google.colab import files
        files.download(csv_path)
        files.download(xlsx_path)
    except ImportError:
        print("(Not in Colab — files saved locally)")
else:
    print("No results to export.")

## 7. Dataset Summary

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

total_videos = len(all_metadata)
processed_count = len([m for m in all_metadata
                       if m.get("processing_status") == "ready"])
unprocessed_count = total_videos - processed_count

print(f"Total videos:       {total_videos}")
print(f"Processed (ready):  {processed_count}")
print(f"Unprocessed:        {unprocessed_count}")

if not results_df.empty:
    total_segments = len(results_df)
    total_duration = results_df["duration_sec"].sum()
    print(f"Total segments:     {total_segments}")
    print(f"Total behavior time: {total_duration:.1f} sec")
    print()

    # Behavior distribution
    behavior_counts = results_df["behavior"].value_counts()
    print("Behavior distribution:")
    for beh, count in behavior_counts.items():
        beh_dur = results_df[results_df["behavior"] == beh]["duration_sec"].sum()
        print(f"  {beh:15s}  {count:4d} segments  {beh_dur:7.1f} sec")

    # Pie chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Video status pie
    status_counts = pd.Series({
        "Processed": processed_count,
        "Unprocessed": unprocessed_count,
    })
    axes[0].pie(status_counts, labels=status_counts.index,
                colors=["#2ecc71", "#e74c3c"],
                autopct="%1.0f%%", startangle=90)
    axes[0].set_title("Video Processing Status")

    # Behavior pie
    colors_beh = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12",
                  "#9b59b6", "#95a5a6", "#1abc9c", "#e67e22"]
    axes[1].pie(behavior_counts, labels=behavior_counts.index,
                colors=colors_beh[:len(behavior_counts)],
                autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Behavior Distribution (all videos)")

    plt.tight_layout()
    plt.show()
else:
    print("\nNo processed results available for summary.")